Objectives
1. Extract meaningful text features from TED Talks.
2. Prepare titles, descriptions, tags and transcripts.
3. Handle missing transcripts without losing useful talks.
4. Create combined text features for vectorisation.
5. Validate and save the engineered dataset.

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Text processing
import re
import unicodedata

# File and directory management
from pathlib import Path

# Reporting
import json

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Libraries imported successfully!")

Libraries imported successfully!


Why are these libraries required?
Library	Purpose
Pandas	Dataset manipulation
NumPy	Numerical operations
Re	Text cleaning using regular expressions
Unicodedata	Unicode normalisation
Pathlib	File and folder management
JSON	Saving configuration and reports

2. Configure project directories

In [2]:
# Automatically locate project root
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT != PROJECT_ROOT.parent:

    input_file = (
        PROJECT_ROOT /
        "Module_1_Data_Collection" /
        "Scraped" /
        "Data" /
        "Processed" /
        "ted_talks_preprocessed.csv"
    )

    if input_file.exists():
        break

    PROJECT_ROOT = PROJECT_ROOT.parent

if not input_file.exists():
    raise FileNotFoundError(
        "Processed TED Talks dataset not found. "
        "Check your project directory."
    )

# Input dataset
INPUT_PATH = input_file

# Module 3 directories
MODULE_DIR = PROJECT_ROOT / "Module_3_Feature_Engineering_Vectorization"

FEATURE_DIR = MODULE_DIR / "features"
OUTPUT_DIR = MODULE_DIR / "outputs"

REPORT_DIR = OUTPUT_DIR / "reports"
STATISTICS_DIR = OUTPUT_DIR / "statistics"

# Create directories
for folder in [
    FEATURE_DIR,
    REPORT_DIR,
    STATISTICS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input dataset:", INPUT_PATH)
print("Feature directory:", FEATURE_DIR)

Project root: C:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation
Input dataset: C:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Module_1_Data_Collection\Scraped\Data\Processed\ted_talks_preprocessed.csv
Feature directory: C:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Module_3_Feature_Engineering_Vectorization\features


3. Load the preprocessed dataset

In [3]:
df = pd.read_csv(
    INPUT_PATH,
    low_memory=False
)

print("Dataset loaded successfully!")

print("\nDataset shape:", df.shape)

print("\nAvailable columns:")
print(df.columns.tolist())

display(df.head())

Dataset loaded successfully!

Dataset shape: (2550, 43)

Available columns:
['talk_id', 'title', 'main_speaker', 'speaker_occupation', 'description', 'tags', 'transcript', 'related_talks', 'ratings', 'views', 'comments', 'duration', 'published_date', 'event', 'url', 'source', 'film_date', 'languages', 'name', 'num_speaker', 'tags_list', 'tag_count', 'tags_text', 'title_clean', 'description_clean', 'transcript_clean', 'speaker_clean', 'published_datetime', 'published_year', 'published_month', 'duration_minutes', 'title_word_count', 'title_char_count', 'description_word_count', 'description_char_count', 'transcript_word_count', 'transcript_char_count', 'has_transcript', 'has_description', 'has_tags', 'content_field_count', 'content_text', 'content_word_count']


,talk_id,title,main_speaker,speaker_occupation,description,tags,transcript,related_talks,ratings,views,comments,duration,published_date,event,url,source,film_date,languages,name,num_speaker,tags_list,tag_count,tags_text,title_clean,description_clean,transcript_clean,speaker_clean,published_datetime,published_year,published_month,duration_minutes,title_word_count,title_char_count,description_word_count,description_char_count,transcript_word_count,transcript_char_count,has_transcript,has_description,has_tags,content_field_count,content_text,content_word_count
0,1,Do schools kill creativity?,Ken Robinson,Author/educator,Sir Ken Robinson makes an entertaining and profoundly moving case for creating an education system that nurtures (ra...,"['children', 'creativity', 'culture', 'dance', 'education', 'parenting', 'teaching']","Good morning. How are you?(Laughter)It's been great, hasn't it? I've been blown away by the whole thing. In fact, I'...","[{'id': 865, 'hero': 'https://pe.tedcdn.com/images/ted/172559_800x600.jpg', 'speaker': 'Ken Robinson', 'title': 'Bri...","[{'id': 7, 'name': 'Funny', 'count': 19645}, {'id': 1, 'name': 'Beautiful', 'count': 4573}, {'id': 9, 'name': 'Ingen...",47227110,4553,1164,1151367060,TED2006,https://www.ted.com/talks/ken_robinson_says_schools_kill_creativity,TED_Public_Archive_CC_BY_NC_SA_4.0,1140825600,60,Ken Robinson: Do schools kill creativity?,1,"[""children"", ""creativity"", ""culture"", ""dance"", ""education"", ""parenting"", ""teaching""]",7,children creativity culture dance education parenting teaching,do schools kill creativity,sir ken robinson makes an entertaining and profoundly moving case for creating an education system that nurtures rat...,good morning how are you laughter it's been great hasn't it i've been blown away by the whole thing in fact i'm leav...,ken robinson,2006-06-27 00:11:00+00:00,2006,6,19.40,4,26,21,146,3163,16800,True,True,True,4,do schools kill creativity sir ken robinson makes an entertaining and profoundly moving case for creating an educati...,3197
1,2,Averting the climate crisis,Al Gore,Climate advocate,"With the same humor and humanity he exuded in ""An Inconvenient Truth,"" Al Gore spells out 15 ways that individuals c...","['alternative energy', 'cars', 'climate change', 'culture', 'environment', 'global issues', 'science', 'sustainabili...","Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extr...","[{'id': 243, 'hero': 'https://pe.tedcdn.com/images/ted/566c14767bd62c5ff760e483c5b16cd2753328cd_2880x1620.jpg', 'spe...","[{'id': 7, 'name': 'Funny', 'count': 544}, {'id': 3, 'name': 'Courageous', 'count': 139}, {'id': 2, 'name': 'Confusi...",3200520,265,977,1151367060,TED2006,https://www.ted.com/talks/al_gore_on_averting_climate_crisis,TED_Public_Archive_CC_BY_NC_SA_4.0,1140825600,43,Al Gore: Averting the climate crisis,1,"[""alternative energy"", ""cars"", ""climate change"", ""culture"", ""environment"", ""global issues"", ""science"", ""sustainabili...",9,alternative energy cars climate change culture environment global issues science sustainability technology,averting the climate crisis,with the same humor and humanity he exuded in an inconvenient truth al gore spells out 15 ways that individuals can ...,thank you so much chris and it's truly a great honor to have the opportunity to come to this stage twice i'm extreme...,al gore,2006-06-27 00:11:00+00:00,2006,6,16.28,4,27,39,227,2155,11461,True,True,True,4,averting the climate crisis with the same humor and humanity he exuded in an inconvenient truth al gore spells out 1...,2212
2,3,Simplicity sells,David Pogue,Technology columnist,"New York Times columnist David Pogue takes aim at technology’s worst interface-design offenders, and provides encour...","['computers', 'entertainment', 'interface design', 'media', 'music', 'performance', 'simplicity', 'software', 'techn...","(Music: ""The Sound of Silence,"" Simon & Garfunkel)Hello voice mail, my ol

Purpose: We are loading the output of Module 2, not the raw dataset. This ensures that our feature engineering starts with the previously cleaned data.
4. Identify important text columns
We need four primary text features:
- Title
- Description
- Tags
- Transcript

In [4]:
def find_column(dataframe, candidates):

    column_lookup = {
        str(col).strip().lower(): col
        for col in dataframe.columns
    }

    for candidate in candidates:

        if candidate.lower() in column_lookup:
            return column_lookup[candidate.lower()]

    return None


column_map = {

    "title": find_column(
        df,
        ["title", "talk_title"]
    ),

    "description": find_column(
        df,
        ["description", "summary", "talk_description"]
    ),

    "tags": find_column(
        df,
        ["tags", "tag", "topics", "topic"]
    ),

    "transcript": find_column(
        df,
        ["transcript", "talk_transcript", "text"]
    )

}

print("Identified columns:")

for field, column in column_map.items():

    print(f"{field}: {column}")


missing_fields = [
    field
    for field, column in column_map.items()
    if column is None
]

print("\nMissing source fields:", missing_fields)

if column_map["title"] is None:

    raise ValueError(
        "Title column is required. "
        "Please check your dataset."
    )

Identified columns:
title: title
description: description
tags: tags
transcript: transcript

Missing source fields: []


Why do this?

Different datasets may use different column names. This mapping makes our notebook more reusable.
5. Understand text feature availability
Before cleaning, analyse how much text information is available.

In [5]:
text_availability = {}

for field, column in column_map.items():

    if column is not None:

        text_availability[field] = (
            df[column]
            .notna()
            .sum()
        )

    else:

        text_availability[field] = 0


availability_df = pd.DataFrame({

    "Feature": text_availability.keys(),

    "Available Records": text_availability.values()

})

availability_df["Missing Records"] = (
    len(df) - availability_df["Available Records"]
)

availability_df["Coverage Percentage"] = (
    availability_df["Available Records"] /
    len(df) * 100
).round(2)

display(availability_df)

,Feature,Available Records,Missing Records,Coverage Percentage
0,title,2550,0,100.00
1,description,2550,0,100.00
2,tags,2550,0,100.00
3,transcript,2464,86,96.63


This helps us understand whether we can depend on transcripts for every TED Talk.

For example, if some talks have no transcripts, the recommendation system must still use their titles, descriptions and tags.
6. Create a text-cleaning function
Now we prepare the text for feature engineering.

We will use conservative cleaning rather than aggressive removal of meaningful words.

In [6]:
def clean_text(value):

    if pd.isna(value):
        return ""

    # Convert to string
    text = str(value)

    # Unicode normalisation
    text = unicodedata.normalize("NFKC", text)

    # Remove HTML tags
    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Convert to lowercase
    text = text.lower()

    # Remove extra whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

Important decision: We are not removing words such as not, no, and never. These words can change the meaning of a TED Talk.

We are also not removing punctuation or numbers indiscriminately.
7. Apply text cleaning to each feature

In [7]:
# Create separate processed text columns

for field in [
    "title",
    "description",
    "tags",
    "transcript"
]:

    source_column = column_map[field]

    if source_column is not None:

        df[f"{field}_processed"] = (
            df[source_column]
            .apply(clean_text)
        )

    else:

        df[f"{field}_processed"] = ""


print("Text preprocessing completed!")

display(
    df[
        [
            "title_processed",
            "description_processed",
            "tags_processed",
            "transcript_processed"
        ]
    ].head()
)

Text preprocessing completed!


,title_processed,description_processed,tags_processed,transcript_processed
0,do schools kill creativity?,sir ken robinson makes an entertaining and profoundly moving case for creating an education system that nurtures (ra...,"['children', 'creativity', 'culture', 'dance', 'education', 'parenting', 'teaching']","good morning. how are you?(laughter)it's been great, hasn't it? i've been blown away by the whole thing. in fact, i'..."
1,averting the climate crisis,"with the same humor and humanity he exuded in ""an inconvenient truth,"" al gore spells out 15 ways that individuals c...","['alternative energy', 'cars', 'climate change', 'culture', 'environment', 'global issues', 'science', 'sustainabili...","thank you so much, chris. and it's truly a great honor to have the opportunity to come to this stage twice; i'm extr..."
2,simplicity sells,"new york times columnist david pogue takes aim at technology’s worst interface-design offenders, and provides encour...","['computers', 'entertainment', 'interface design', 'media', 'music', 'performance', 'simplicity', 'software', 'techn...","(music: ""the sound of silence,"" simon & garfunkel)hello voice mail, my old friend.(laughter)i've called for tech sup..."
3,greening the ghetto,"in an emotionally charged talk, macarthur-winning activist majora carter details her fight for environmental justice...","['macarthur grant', 'activism', 'business', 'cities', 'environment', 'green', 'inequality', 'politics', 'pollution']",if you're here today — and i'm very happy that you are — you've all heard about how sustainable development will sav...
4,the best stats you've ever seen,"you've never seen data presented like this. with the drama and urgency of a sportscaster, statistics guru hans rosli...","['africa', 'asia', 'google', 'demo', 'economics', 'global development', 'global issues', 'health', 'math', 'statisti...","about 10 years ago, i took on the task to teach global development to swedish undergraduate students. that was after..."


We retain the original columns and create new processed columns. This means we can compare original and processed data whenever necessary.
8. Feature engineering – combine text fields
This is one of the most important steps.

A TED Talk's meaning is not represented by its transcript alone.

For example:
- Title: The power of vulnerability
- Description: A talk about human connection and emotional openness.
- Tags: Psychology, relationships, personal growth.
- Transcript: The complete speech.
Combining these fields gives the recommendation system more information.

In [8]:
def combine_text_fields(row):

    fields = [

        row["title_processed"],

        row["description_processed"],

        row["tags_processed"],

        row["transcript_processed"]

    ]

    # Include only non-empty fields
    available_fields = [
        text for text in fields
        if text.strip()
    ]

    return " ".join(available_fields)


df["combined_text"] = df.apply(
    combine_text_fields,
    axis=1
)

display(
    df[
        [
            "title_processed",
            "tags_processed",
            "combined_text"
        ]
    ].head()
)

,title_processed,tags_processed,combined_text
0,do schools kill creativity?,"['children', 'creativity', 'culture', 'dance', 'education', 'parenting', 'teaching']",do schools kill creativity? sir ken robinson makes an entertaining and profoundly moving case for creating an educat...
1,averting the climate crisis,"['alternative energy', 'cars', 'climate change', 'culture', 'environment', 'global issues', 'science', 'sustainabili...","averting the climate crisis with the same humor and humanity he exuded in ""an inconvenient truth,"" al gore spells ou..."
2,simplicity sells,"['computers', 'entertainment', 'interface design', 'media', 'music', 'performance', 'simplicity', 'software', 'techn...","simplicity sells new york times columnist david pogue takes aim at technology’s worst interface-design offenders, an..."
3,greening the ghetto,"['macarthur grant', 'activism', 'business', 'cities', 'environment', 'green', 'inequality', 'politics', 'pollution']","greening the ghetto in an emotionally charged talk, macarthur-winning activist majora carter details her fight for e..."
4,the best stats you've ever seen,"['africa', 'asia', 'google', 'demo', 'economics', 'global development', 'global issues', 'health', 'math', 'statisti...",the best stats you've ever seen you've never seen data presented like this. with the drama and urgency of a sportsca...


Why combine these fields?

The title provides the central topic, tags provide subject categories, descriptions provide summaries, and transcripts provide detailed context.

We will experiment with field weighting in Notebook 02 rather than artificially repeating text here.
9. Handle missing transcripts
A talk without a transcript can still be recommended using its title, description and tags.

Let's create a fallback content representation.

In [9]:
df["content_text_fallback"] = df.apply(

    lambda row: " ".join(

        text for text in [

            row["title_processed"],

            row["description_processed"],

            row["tags_processed"]

        ]

        if text.strip()

    ),

    axis=1

)

# Identify transcript availability

df["has_transcript"] = (
    df["transcript_processed"].str.len() > 0
)

df["uses_fallback"] = (
    ~df["has_transcript"]
)

print("Talks with transcripts:",
      df["has_transcript"].sum())

print("Talks requiring fallback:",
      df["uses_fallback"].sum())

Talks with transcripts: 2464
Talks requiring fallback: 86


This avoids unnecessarily discarding useful TED Talks.
10. Generate text statistics
We should measure the length of each feature.

In [10]:
text_columns = [

    "title_processed",

    "description_processed",

    "tags_processed",

    "transcript_processed",

    "combined_text"

]

for column in text_columns:

    df[f"{column}_word_count"] = (
        df[column]
        .str.split()
        .str.len()
        .fillna(0)
        .astype(int)
    )

    df[f"{column}_char_count"] = (
        df[column]
        .str.len()
    )


display(

    df[
        [
            "title_processed_word_count",
            "description_processed_word_count",
            "tags_processed_word_count",
            "transcript_processed_word_count",
            "combined_text_word_count"
        ]
    ].describe()

)

,title_processed_word_count,description_processed_word_count,tags_processed_word_count,transcript_processed_word_count,combined_text_word_count
count,2550.000000,2550.000000,2550.000000,2550.000000,2550.000000
mean,6.217255,51.850588,8.641961,1971.421569,2038.131373
std,2.256731,18.112417,5.121287,1009.507455,1010.329312
min,1.000000,9.000000,1.000000,0.000000,24.000000
25%,5.000000,39.000000,5.000000,1235.750000,1297.750000
50%,6.000000,49.000000,7.000000,1982.500000,2052.500000
75%,8.000000,63.000000,10.000000,2680.500000,2743.500000
max,16.000000,130.000000,38.000000,9044.000000,9159.000000


These statistics will help us understand how large our documents are before vectorisation.
11. Validate engineered features

In [11]:
validation_report = {

    "total_records": int(len(df)),

    "records_with_title": int(
        df["title_processed"].str.len().gt(0).sum()
    ),

    "records_with_description": int(
        df["description_processed"].str.len().gt(0).sum()
    ),

    "records_with_tags": int(
        df["tags_processed"].str.len().gt(0).sum()
    ),

    "records_with_transcript": int(
        df["has_transcript"].sum()
    ),

    "records_using_fallback": int(
        df["uses_fallback"].sum()
    ),

    "records_with_combined_text": int(
        df["combined_text"].str.len().gt(0).sum()
    ),

    "records_with_empty_combined_text": int(
        df["combined_text"].str.len().eq(0).sum()
    )

}

print("FEATURE ENGINEERING VALIDATION")

for key, value in validation_report.items():

    print(f"{key}: {value}")

FEATURE ENGINEERING VALIDATION
total_records: 2550
records_with_title: 2550
records_with_description: 2550
records_with_tags: 2550
records_with_transcript: 2464
records_using_fallback: 86
records_with_combined_text: 2550
records_with_empty_combined_text: 0


12. Save engineered dataset
We must save the engineered data separately.

In [12]:
FEATURE_PATH = (
    FEATURE_DIR /
    "ted_talks_features.csv"
)

REPORT_PATH = (
    REPORT_DIR /
    "feature_engineering_validation.json"
)

COVERAGE_PATH = (
    STATISTICS_DIR /
    "text_feature_coverage.csv"
)

# Save feature dataset
df.to_csv(
    FEATURE_PATH,
    index=False,
    encoding="utf-8"
)

# Save validation report
with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        validation_report,
        file,
        indent=4
    )

# Save feature availability statistics
availability_df.to_csv(
    COVERAGE_PATH,
    index=False
)

print("All outputs saved successfully!")

print("\nFeature dataset:", FEATURE_PATH)

print("Validation report:", REPORT_PATH)

print("Coverage report:", COVERAGE_PATH)

All outputs saved successfully!

Feature dataset: C:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Module_3_Feature_Engineering_Vectorization\features\ted_talks_features.csv
Validation report: C:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Module_3_Feature_Engineering_Vectorization\outputs\reports\feature_engineering_validation.json
Coverage report: C:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Module_3_Feature_Engineering_Vectorization\outputs\statistics\text_feature_coverage.csv


13. Final output verification

In [13]:
output_files = [

    FEATURE_PATH,

    REPORT_PATH,

    COVERAGE_PATH

]

print("FINAL OUTPUT VERIFICATION")

for file in output_files:

    print(
        file.name,
        "->",
        "SAVED" if file.exists() else "NOT FOUND"
    )

    if file.exists():

        print(
            "File size:",
            f"{file.stat().st_size:,} bytes"
        )

FINAL OUTPUT VERIFICATION
ted_talks_features.csv -> SAVED
File size: 153,615,604 bytes
feature_engineering_validation.json -> SAVED
File size: 293 bytes
text_feature_coverage.csv -> SAVED
File size: 154 bytes
